# Demo of the knime2galaxy workflow
This notebooks guides you through the most important steps during the KNIME to Galaxy conversion. Basically, it explains how the .knwf file is translated to a valid .ga file using the **translate_knime_to_galaxy** function.

In [ ]:
from imaging_knime_to_galaxy.translate import translate_knime_to_galaxy
from pathlib import Path

data_folder = Path("../data").resolve()

workflow = translate_knime_to_galaxy(
    knwf_path=data_folder / "file_to_translate" / "2025_03_2D_spot_detection.knwf",
    tools_metadata_path=data_folder / "tools_metadata.json",
    translation_table_path=data_folder / "translation_table.yml",
    workflow_examples_yml_path=data_folder / "workflow_translation_table.yml",
    output_galaxy_workflow_path=data_folder / "output_file" / "test1_knime2galaxy_output.ga",
    input_workflow_path =data_folder / "input_workflows.ga",
    vector_store_path =data_folder/ "vector_store.npz",

)

# 1. Input Variables
Lets take a look at the function definition to understand the process:

```
def translate_knime_to_galaxy(
        knwf_path: str,
        tools_metadata_path: str,
        translation_table_path: str,
        workflow_examples_yml_path: str,
        output_galaxy_workflow_path: str,
        input_workflow_path: str,
        vector_store_path: str,
):
```

### knwf_path: str
-> Path to the .knwf file to be translated 

### tools_metadata_path: str
-> Path to the .json file which stores all galaxy tools (n=3696) from the toolshed

### translation_table_path: str
-> Path to the .yml file which stores 7 examples of KNIME->GALAXY **tool mapping** (+ additional mapping to Python)

### workflow_examples_yml_path: str
-> Path to the .yml file which stores 2 examples of KNIME->GALAXY **workflow mapping**

### output_galaxy_workflow_path: str
-> Output path for the newly generated .ga file

### input_workflow_path: str
-> Example .ga workflow file

### vector_store_path: str
-> Path to the cached Vector Store embeddings

# 2. Load and build tool meta data

```
    ...
    meta_data = load_tools_metadata(tools_metadata_path)
    texts, metas = build_all_docs(meta_data)
    ...
```

Loads the tools .json file holding their metadata like version, description, tool-ID, ... .

Afterwards, all gets converted to a clear and uniformly structured text and dictionary like output.

# 3. Vector Storage
```
    ...
    vector_store = VectorStore(embed_fn= embed, texts=texts, metadatas=metas)
    ...
```

### Embed (using "Qwen/Qwen3-Embedding-4B") Tools
- text ("name", "description", "repo_description", "repo_long_description", and "detailed_description_generated")
- metadata ("owner", "repo_name", "tool_id", "version", and "guid")

### Store their vectors 
- NumPy matrix with dimension: n_documents × embedding_dimension

### Compare query against stored embeddings
- similarity using dot product
- find the examples that are most similar to the query (= KNIME file)


# 4. KNIME file processing
```
    ...
    knime_nodes = collect_knime_node_files(knwf_path=knwf_path)
    workflow_content = collect_workflow_file(knwf_path)
    node_examples = build_translation_examples(yaml_path=translation_table_path)
    knime_nodes_str = convert_knime_dict_to_string(knime_nodes)
    ...
```

### Collect the nodes from the KNIME workflow

### Collect the workflow itself from the .knwf file

### Build a string that holds KNIME to Galaxy conversion examples

### Convert KNIME nodes to a string representation

# 5. Prompting the LLM
```
    ...
    summary_task = build_summary_prompt(knime_nodes_str, workflow_content)
    workflow_examples = build_workflow_examples(yaml_path=workflow_examples_yml_path)

    full_summary_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{summary_task}"
    summary_answer = prompt_scadsai_llm(message= full_summary_prompt)
    description_task = build_description_task_prompt(knime_nodes_str, workflow_content, summary_answer)
    full_description_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{description_task}"
    description = prompt_scadsai_llm(message= full_description_prompt)
    ...
```

### Enhance the task with the given .knwf file content and workflow examples

### Summary prompt
- Node Examples: KNIME->GALAXY tool mapping table
- Workflow Examples: KNIME->GALAXY workflow mapping table
- Summary Task: specific KNIME workflow content from the file we want to translate

###  Final prompt
- Build a description using LLM: "For each node (step) in the workflow, write a 3 to 5 sentences description of what that node does, using simple technical verbs (e.g. trim, filter, convert, normalize, cluster)."
- use this describtion + node example + workflow example to generate a valid answer

# 6. Vector Search and final .ga Generation
```
    ...
    hits = search_store_for_hits(description, vector_store)
    input_tools = load_galaxy_input_tools()
    task = build_task_prompt(knime_nodes_str, workflow_content, summary_answer, hits, input_tools)
    full_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{task}"

    answer = prompt_scadsai_llm(message= full_prompt)
    json_object = parse_answer_as_json(answer)
    replace_uuid(json_object)
    save_answer_to_file(json_object, output_path=output_galaxy_workflow_path)
```

### Search vector storage for the most similar content (tools)

### Enhance the prompt with a "empty"/example .ga workflow file

### Prompt the model to build a .ga workflow file that has the same functionality as the given .knwf file

### check if its valid JSON

### Generate new random UUID for the workflow

### Save the .ga file